In [ ]:
"""
TIER 2 (LLM-based identifier extraction).


To keep this affordable, NOT every entity is sent to the LLM. A cheap
regex pre-filter first flags entities that MIGHT contain an identifier
(anything with a digit, a parenthetical, or a dash), a wide net meant
to catch everything worth checking, false positives here are fine and
cheap, false negatives would mean missing a real ID, so the filter is
kept deliberately loose. Only flagged entities go to the LLM.

For each flagged entity, the LLM returns:
  identifier_code       - the specific ID string found, or null
  has_distinguishing_id - true if that ID actually distinguishes this
                           entity from similar ones in the same paper

Entities sharing a DOI and the same identifier_code are proposed as
merge candidates (same state-modifier conflict check as before, e.g.
"gel" vs no "gel" still blocks a merge even if the code matches).

Requires OPENAI_API_KEY.
Designed for Jupyter/Colab execution. No __main__ guard.
"""

import os
import re
import json
import time
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

# ---------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------
TRIPLES_PATH = "expanded_triples.xlsx"  # output of consolidate_resolutions.py, NOT the raw triples file
SOURCE_COL = "expanded_source"  # abbreviation-resolved entity text, not the raw "source" column
TARGET_COL = "expanded_target"
DOI_COL = "doi"

LLM_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
MIN_SIMILARITY_SAFETY_CHECK = 0.90

OUTPUT_XLSX = "tier2_llm_code_extraction_review.xlsx"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

STATE_MODIFIER_KEYWORDS = [
    "gel", "commercial", "native", "concentrate", "hydrolysate",
    "hydrolyzate", "complex", "conjugate", "aggregate", "denatured",
]


def extract_state_modifiers(entity_name):
    name = str(entity_name).lower()
    return {kw for kw in STATE_MODIFIER_KEYWORDS if re.search(rf"\b{kw}\b", name)}


# ---------------------------------------------------------------
# STEP 1: CHEAP REGEX PRE-FILTER (wide net, not precise, just cheap)
# ---------------------------------------------------------------
def might_contain_identifier(entity_name):
    """
    Loose check: does this entity name plausibly contain an ID?
    True if it has a digit, a parenthetical, or a dash. This is
    deliberately over-inclusive, missing a real candidate here means
    it never reaches the LLM at all, so err toward flagging too much
    rather than too little.
    """
    name = str(entity_name)
    return bool(re.search(r"\d", name) or re.search(r"[()]", name) or re.search(r"[-\u2010-\u2015]", name))


# ---------------------------------------------------------------
# LOAD ENTITIES
# ---------------------------------------------------------------
def load_triples(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    elif path.endswith(".xlsx"):
        return pd.read_excel(path)
    elif path.endswith(".csv"):
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")


df = load_triples(TRIPLES_PATH)
source_long = df[[SOURCE_COL, DOI_COL]].rename(columns={SOURCE_COL: "entity"})
target_long = df[[TARGET_COL, DOI_COL]].rename(columns={TARGET_COL: "entity"})
long_df = pd.concat([source_long, target_long])
long_df["entity"] = long_df["entity"].astype(str).str.strip()
long_df = long_df.drop_duplicates()

print(f"Total unique (entity, DOI) pairs: {len(long_df)}")

long_df["might_have_id"] = long_df["entity"].map(might_contain_identifier)
candidates = long_df[long_df["might_have_id"]].copy()
print(f"Flagged by cheap pre-filter (sent to LLM): {len(candidates)}")

# ---------------------------------------------------------------
# STEP 2: LLM EXTRACTION (only on flagged candidates)
# ---------------------------------------------------------------
EXTRACTION_PROMPT = """Entity name from a plant protein research paper: "{entity}"

Does this name contain a specific sample/genotype/variant identifier code
(e.g. "SP-10", "N16-10044", "SPI D", "sample A")? Generic words like
"isolate", "protein", "gel" are NOT identifier codes.

Respond ONLY with JSON, no markdown fences:
{{"identifier_code": "..." or null, "has_distinguishing_id": true/false}}
"""


def extract_identifier(entity_name):
    prompt = EXTRACTION_PROMPT.format(entity=entity_name)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    time.sleep(0.1)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"identifier_code": None, "has_distinguishing_id": False}


extraction_results = []
for entity in candidates["entity"].unique():
    result = extract_identifier(entity)
    extraction_results.append({
        "entity": entity,
        "identifier_code": result.get("identifier_code"),
        "has_distinguishing_id": result.get("has_distinguishing_id"),
    })

extraction_df = pd.DataFrame(extraction_results)
print(f"\nExtraction complete: {extraction_df['has_distinguishing_id'].sum()} entities "
      f"confirmed to have a distinguishing ID")

# join identifier codes back onto the (entity, doi) pairs
coded = long_df.merge(extraction_df, on="entity", how="left")
coded = coded[coded["has_distinguishing_id"] == True].copy()  # noqa: E712

# ---------------------------------------------------------------
# STEP 3: GROUP BY (DOI, IDENTIFIER_CODE), STATE-CONFLICT CHECK,
# SIMILARITY SAFETY CHECK (same logic as the regex version)
# ---------------------------------------------------------------
coded_entities = sorted(coded["entity"].unique())


def embed_batch(texts, batch_size=200):
    if not texts:
        return np.array([])
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
        time.sleep(0.2)
    return np.array(vectors)


embeddings = embed_batch(coded_entities)
idx_of = {e: i for i, e in enumerate(coded_entities)}
sim_matrix = cosine_similarity(embeddings) if len(coded_entities) else np.array([[]])

merge_rows, skipped_state_conflict, skipped_low_similarity = [], [], []

for (doi, code), group in coded.groupby([DOI_COL, "identifier_code"]):
    members = sorted(group["entity"].unique())
    if len(members) < 2:
        continue

    state_sets = {m: extract_state_modifiers(m) for m in members}
    if len(set(frozenset(s) for s in state_sets.values())) > 1:
        skipped_state_conflict.append({
            "doi": doi, "code": code, "members": "; ".join(members),
            "state_modifiers_found": "; ".join(
                f"{m}={sorted(state_sets[m]) if state_sets[m] else 'none'}" for m in members
            ),
        })
        continue

    pairwise_sims = [
        sim_matrix[idx_of[members[i]], idx_of[members[j]]]
        for i in range(len(members)) for j in range(i + 1, len(members))
        if members[i] in idx_of and members[j] in idx_of
    ]
    if pairwise_sims and min(pairwise_sims) < MIN_SIMILARITY_SAFETY_CHECK:
        skipped_low_similarity.append({"doi": doi, "code": code, "members": "; ".join(members),
                                         "min_similarity": round(min(pairwise_sims), 3)})
        continue

    canonical = max(members, key=len)
    for m in members:
        merge_rows.append({"doi": doi, "code": code, "raw_name": m, "proposed_canonical_name": canonical})

merge_df = pd.DataFrame(merge_rows)
state_conflict_df = pd.DataFrame(skipped_state_conflict)
skipped_df = pd.DataFrame(skipped_low_similarity)

print(f"\nProposed auto-merges: {len(merge_df)} entities")
print(f"Skipped, state modifier conflict: {len(state_conflict_df)} groups")
print(f"Skipped, similarity too low: {len(skipped_df)} groups")

# ---------------------------------------------------------------
# SAVE
# ---------------------------------------------------------------
readme_rows = [
    "HOW TO READ THIS FILE",
    "",
    "- llm_extractions: every entity flagged by the cheap regex pre-filter as POSSIBLY containing a code was sent to the LLM. This sheet is the raw extraction result: identifier_code (what the LLM found, or blank) and has_distinguishing_id (True/False, whether that code actually distinguishes this entity from similar ones in the same paper). This is a first pass, not a merge decision, nothing here has been merged yet.",
    "",
    "- proposed_auto_merges: built FROM the has_distinguishing_id = True rows in llm_extractions. Those entities are grouped by same paper (DOI) + same identifier_code, then automatically reviewed: do members disagree on a real material-state word like \"gel\" or \"commercial\"? Is the text similarity between them too low to trust? Only groups that pass BOTH checks land here as confident, ready-to-merge entities. If this sheet is empty, it does not mean nothing was found, it means every candidate group got caught by one of the two review checks below.",
    "",
    "- skipped_state_conflict: candidate merge groups (same DOI + same code) that got BLOCKED because members disagree on a real material-state word (e.g. \"gel\" appears on one entity but not the other sharing the same code). These are NOT auto-merged, same code plus a different state usually means genuinely different samples (e.g. the raw isolate vs. its gelled form). Needs a manual or tier 3 review before deciding either way.",
    "",
    "- skipped_low_similarity: candidate merge groups where the code matched but overall text similarity between members fell below the trust threshold. The shared code could be coincidence rather than real evidence of the same object. Needs a manual or tier 3 review before deciding either way.",
]
readme_df = pd.DataFrame({"": readme_rows})

with pd.ExcelWriter(OUTPUT_XLSX) as writer:
    readme_df.to_excel(writer, sheet_name="READ_ME_FIRST", index=False)
    extraction_df.to_excel(writer, sheet_name="llm_extractions", index=False)
    merge_df.to_excel(writer, sheet_name="proposed_auto_merges", index=False)
    state_conflict_df.to_excel(writer, sheet_name="skipped_state_conflict", index=False)
    skipped_df.to_excel(writer, sheet_name="skipped_low_similarity", index=False)

print(f"\nSaved to {OUTPUT_XLSX}")

Total unique (entity, DOI) pairs: 6010
Flagged by cheap pre-filter (sent to LLM): 1007

Extraction complete: 98 entities confirmed to have a distinguishing ID

Proposed auto-merges: 0 entities
Skipped, state modifier conflict: 20 groups
Skipped, similarity too low: 1 groups

Saved to tier2_llm_code_extraction_review.xlsx
